# LeetCode 261 — Graph Valid Tree bằng Python

Link đề: https://leetcode.com/problems/graph-valid-tree/ · Độ khó: **Medium** (Premium) · Chủ đề: Graph, Union Find, DFS, BFS

Skeleton LeetCode:

```python
class Solution:
    def validTree(self, n: int, edges: List[List[int]]) -> bool:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đồ thị phải liên thông và **mọi cạnh là cầu**, tính liên thông bằng bao đóng bắc cầu Floyd–Warshall).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `n` nút `0..n-1` và danh sách cạnh **vô hướng**, trả về `true` nếu các cạnh đó tạo thành **một cây** (đồ thị **liên thông** và **không có chu trình**).
2. Nắm **ba cách phát biểu tương đương** của "cây" với `n` nút: (a) liên thông và không chu trình; (b) **liên thông và có đúng `n - 1` cạnh**; (c) **không chu trình và có đúng `n - 1` cạnh**.
3. Cài đặt đồ thị bằng **danh sách kề (adjacency list)** và làm quen **DFS**, **BFS**, **Union-Find (DSU)** với **nén đường đi** và cách phát hiện chu trình bằng DSU.
4. Đi từ **"liên thông và mọi cạnh là cầu"** (vét cạn) → **Union-Find** → **DFS có theo dõi cha** → **BFS + đếm cạnh** → **bóc lá (leaf peeling)**.
5. Tránh các lỗi kinh điển: chỉ kiểm tra **chu trình** mà quên **liên thông** (hoặc ngược lại), quên **loại cạnh về cha** khi DFS đồ thị vô hướng, quên xây **hai chiều** cạnh, quên `visited == n`, và **tràn đệ quy** khi chuỗi dài.

## 2. Đề bài & ví dụ

Bạn có một đồ thị gồm `n` nút được đánh nhãn từ `0` đến `n - 1`. Cho số nguyên `n` và danh sách `edges` với `edges[i] = [a_i, b_i]` nghĩa là có **cạnh vô hướng** giữa nút `a_i` và `b_i`.

Trả về `true` nếu các cạnh của đồ thị tạo thành một **cây hợp lệ**, ngược lại `false`.

**Ví dụ 1**

- Input: `n = 5, edges = [[0,1],[0,2],[0,3],[1,4]]`
- Output: `true`

**Ví dụ 2**

- Input: `n = 5, edges = [[0,1],[1,2],[2,3],[1,3],[1,4]]`
- Output: `false`
- Giải thích: các nút `1, 2, 3` tạo chu trình `1-2-3-1`.

**Ràng buộc:** `1 <= n <= 2000`; `0 <= edges.length <= 5000`; `edges[i].length == 2`; `0 <= a_i, b_i < n`; `a_i != b_i`; **không** có cạnh lặp và **không** có khuyên (self-loop).

In [1]:
from typing import List
from collections import deque, defaultdict
from itertools import combinations
import random
import time

examples = [
    (5, [[0, 1], [0, 2], [0, 3], [1, 4]], True),
    (5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]], False),
]
for n, edges, expected in examples:
    print(f"n={n}, edges={edges} -> kỳ vọng {expected}")

n=5, edges=[[0, 1], [0, 2], [0, 3], [1, 4]] -> kỳ vọng True
n=5, edges=[[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Đồ thị vô hướng và danh sách kề

**Đồ thị** gồm các **nút (đỉnh)** và **cạnh**. Vô hướng: cạnh `[a, b]` nối `a` với `b` theo **cả hai chiều**. **Danh sách kề** `adj[v]` là danh sách các nút kề với `v`. Với cạnh `[a, b]` phải thêm **cả** `adj[a].append(b)` **và** `adj[b].append(a)`.
#### Code demo

In [2]:
n = 5
edges = [[0, 1], [0, 2], [0, 3], [1, 4]]
adj = [[] for _ in range(n)]
for a, b in edges:
    adj[a].append(b)
    adj[b].append(a)                     # vô hướng: thêm hai chiều
for v in range(n):
    print(f"adj[{v}] = {adj[v]}")
print("bậc của nút 0:", len(adj[0]))

adj[0] = [1, 2, 3]
adj[1] = [0, 4]
adj[2] = [0]
adj[3] = [0]
adj[4] = [1]
bậc của nút 0: 3


### 3.2. Cây là gì? Ba cách phát biểu tương đương

Đồ thị vô hướng có `n` nút là **cây** khi thỏa một (và khi đó thỏa cả ba) điều kiện tương đương:

1. **liên thông** và **không có chu trình**;
2. **liên thông** và có đúng **`n - 1` cạnh**;
3. **không có chu trình** và có đúng **`n - 1` cạnh**.

Trực giác: cây là đồ thị liên thông **"vừa đủ"** cạnh — bớt một cạnh thì mất liên thông, thêm một cạnh thì sinh chu trình. Nút đơn `n = 1` không cạnh cũng là cây.
#### Code demo

In [3]:
for name, (nodes, edge_list) in {
    "đường đi 0-1-2": (3, [[0, 1], [1, 2]]),
    "tam giác (có chu trình)": (3, [[0, 1], [1, 2], [0, 2]]),
    "hai đoạn rời (không liên thông)": (4, [[0, 1], [2, 3]]),
}.items():
    print(f"{name}: n={nodes}, số cạnh={len(edge_list)}, n-1={nodes - 1}")

đường đi 0-1-2: n=3, số cạnh=2, n-1=2
tam giác (có chu trình): n=3, số cạnh=3, n-1=2
hai đoạn rời (không liên thông): n=4, số cạnh=2, n-1=3


### 3.3. DFS/BFS và `visited`

**DFS** (duyệt theo chiều sâu) đi hết một nhánh rồi quay lại; **BFS** (theo chiều rộng) duyệt theo từng lớp. Cả hai dùng tập/mảng **`visited`** để không thăm lại. Từ nút `0`, số nút thăm được là **kích thước thành phần liên thông chứa `0`**; đồ thị liên thông ⟺ thăm được đủ `n` nút.
#### Code demo

In [4]:
def reachable_count(n, adj, start=0):
    visited = [False] * n
    visited[start] = True
    stack = [start]
    count = 1
    while stack:
        v = stack.pop()
        for u in adj[v]:
            if not visited[u]:
                visited[u] = True
                count += 1
                stack.append(u)
    return count


print(reachable_count(5, adj))                         # 5: liên thông
adj2 = [[1], [0], [3], [2]]                            # hai đoạn rời
print(reachable_count(4, adj2))                        # 2 < 4: không liên thông

5
2


### 3.4. Phát hiện chu trình trong đồ thị **vô hướng** khi DFS

Khi đi từ `v` sang `u` qua cạnh `(v, u)`, trong `adj[u]` có `v` (vì vô hướng). **Không được coi** việc "nhìn thấy `v` từ `u`" là chu trình: đó chỉ là **cạnh vừa đi qua**. Ta lưu **cha** `parent[u] = v`; khi duyệt hàng xóm `w` của `u`: nếu `w == parent[u]` thì bỏ qua; nếu `w` **đã thăm** (và không phải cha) thì có **chu trình**. (Do đồ thị không có cạnh lặp nên so sánh với cha là đủ.)
#### Code demo

In [5]:
def has_cycle_dfs(n, adj):
    visited = [False] * n
    for s in range(n):
        if visited[s]:
            continue
        visited[s] = True
        stack = [(s, -1)]                              # (nút, cha)
        while stack:
            v, parent = stack.pop()
            for u in adj[v]:
                if u == parent:
                    continue                           # cạnh quay về cha: bỏ qua
                if visited[u]:
                    return True                        # gặp nút đã thăm khác cha: chu trình
                visited[u] = True
                stack.append((u, v))
    return False


tri = [[1, 2], [0, 2], [0, 1]]
print(has_cycle_dfs(3, tri), has_cycle_dfs(5, adj))    # True False

True False


### 3.5. Union-Find (DSU)

**Union-Find** quản lý các **tập rời nhau**: `find(x)` trả **đại diện (gốc)** của tập chứa `x`; `union(a, b)` gộp hai tập. Mảng `parent[x]` trỏ lên cha, gốc trỏ về chính nó. **Nén đường đi (path compression)** làm `find` gần `O(1)`. Phát hiện chu trình bằng DSU: với mỗi cạnh `(a, b)`, nếu `find(a) == find(b)` thì `a`, `b` **đã liên thông** nên thêm cạnh này tạo **chu trình**; ngược lại `union`.
#### Code demo

In [6]:
parent = list(range(6))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]                  # nén đường đi (path halving)
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False                                   # đã cùng tập: nối thêm sẽ tạo chu trình
    parent[ra] = rb
    return True


print(union(0, 1), union(1, 2), union(0, 2))           # True True False (0-1-2 đã liên thông)
print([find(i) for i in range(6)])

True True False
[2, 2, 2, 3, 4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `n = 5`, `edges = [[0,1],[0,2],[0,3],[1,4]]`. Có `4 = n - 1` cạnh. Từ nút `0` đi được tới `1, 2, 3` và qua `1` tới `4`: thăm đủ `5` nút ⇒ liên thông. `n - 1` cạnh + liên thông ⇒ **cây** ⇒ `true`.

Ví dụ 2: `edges = [[0,1],[1,2],[2,3],[1,3],[1,4]]` có `5` cạnh `> n - 1 = 4` ⇒ **chắc chắn** có chu trình (`1-2-3-1`) ⇒ `false`.

Hai lỗi người mới hay mắc, ví dụ:

- `n = 4`, `edges = [[0,1],[2,3]]`: **không có chu trình** nhưng **không liên thông** ⇒ không phải cây (chỉ `2 < n - 1 = 3` cạnh).
- `n = 3`, `edges = [[0,1],[1,2],[0,2]]`: **liên thông** nhưng **có chu trình** (`3 > n - 1 = 2` cạnh).

**Cách nghĩ:** cây = đồ thị liên thông tối thiểu. Kiểm tra bằng **đếm cạnh** (`n - 1`) kết hợp một trong hai: **liên thông** hoặc **không chu trình**.

## 5. Cách 1 — Brute Force / baseline: liên thông và mọi cạnh đều là cầu

**Ý tưởng.** Một đồ thị liên thông là cây ⟺ **bỏ đi bất kỳ cạnh nào cũng làm mất liên thông** (mọi cạnh là **cầu**). Thuật toán: (1) kiểm tra đồ thị liên thông bằng BFS; (2) với từng cạnh, bỏ cạnh đó rồi BFS lại: nếu vẫn liên thông thì cạnh đó nằm trên chu trình ⇒ không phải cây.
#### Code demo

In [7]:
def connected(n, edge_list):
    adj = [[] for _ in range(n)]
    for a, b in edge_list:
        adj[a].append(b)
        adj[b].append(a)
    seen = {0}
    queue = deque([0])
    while queue:
        v = queue.popleft()
        for u in adj[v]:
            if u not in seen:
                seen.add(u)
                queue.append(u)
    return len(seen) == n


def valid_tree_bruteforce(n, edges):
    if not connected(n, edges):
        return False
    for idx in range(len(edges)):
        without = edges[:idx] + edges[idx + 1:]
        if connected(n, without):                      # bỏ cạnh mà vẫn liên thông: cạnh thuộc chu trình
            return False
    return True


for n, edges, expected in examples:
    print(n, edges, "->", valid_tree_bruteforce(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [0, 2], [0, 3], [1, 4]] -> True (kỳ vọng True )
5 [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `V = n`, `E = len(edges)`: một lần BFS `O(V + E)`; thực hiện `E + 1` lần ⇒ **Time `O(E·(V + E))`**; với `E = 5000`, `V = 2000` là `~3.5·10^7` phép — chậm nhưng khả thi; **Space `O(V + E)`**.

**Nút thắt.** Ta lặp BFS cho **từng cạnh**. Định lý "cây ⟺ liên thông và `n - 1` cạnh" cho phép bỏ hẳn vòng lặp đó: chỉ cần **đếm cạnh** và **một lần** kiểm tra liên thông (hoặc DSU).

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý (đủ dùng):** đồ thị vô hướng `n` nút là cây ⟺ có đúng `n - 1` cạnh **và** liên thông ⟺ có đúng `n - 1` cạnh **và** không chu trình.

*Chứng minh sơ lược:* một đồ thị liên thông `n` nút cần **ít nhất** `n - 1` cạnh (mỗi cạnh gộp tối đa hai thành phần thành một); nếu có đúng `n - 1` cạnh thì mỗi cạnh đều **gộp** hai thành phần khác nhau (không có cạnh nào "thừa" tạo chu trình). Ngược lại cây không chu trình có `n - 1` cạnh.

**Vậy:** (1) nếu `len(edges) != n - 1` ⇒ `false` ngay; (2) nếu bằng `n - 1`, chỉ cần kiểm tra **liên thông** (bằng DSU/DFS/BFS) **hoặc** **không chu trình**.

Với Union-Find: duyệt cạnh, nếu hai đầu đã cùng tập ⇒ chu trình ⇒ `false`; nếu không gặp chu trình và có `n - 1` cạnh thì cuối cùng chỉ còn **1** thành phần (vì `n` nút, mỗi cạnh giảm số thành phần đi 1).

## 7. Các cách tiếp theo

### Cách 2 — Union-Find (cách nộp chuẩn)

**Giải thích.** Khởi tạo `n` tập đơn; với mỗi cạnh `(a, b)`: nếu `find(a) == find(b)` ⇒ chu trình ⇒ `False`; ngược lại `union`, giảm bộ đếm thành phần `n -= 1`. Cuối cùng trả `components == 1`.
#### Code demo

In [8]:
def valid_tree_dsu(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:                          # find lặp (không đệ quy) + nén đường đi kiểu path halving
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    components = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return False                               # a, b đã liên thông: cạnh này tạo chu trình
        parent[ra] = rb
        components -= 1
    return components == 1


for n, edges, expected in examples:
    print(n, edges, "->", valid_tree_dsu(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [0, 2], [0, 3], [1, 4]] -> True (kỳ vọng True )
5 [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> False (kỳ vọng False )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def validTree(self, n: int, edges: List[List[int]]) -> bool:
        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        p = list(range(n))
        for a, b in edges:
            pa, pb = find(a), find(b)
            if pa == pb:
                return False
            p[pa] = pb
            n -= 1
        return n == 1


print(Solution().validTree(5, [[0, 1], [0, 2], [0, 3], [1, 4]]))   # True

True


**Độ phức tạp Cách 2:** **Time `O(E · α(V))`** (≈ `O(E)`; `α` là hàm Ackermann nghịch đảo, gần như hằng), **Space `O(V)`**.

### Cách 3 — DFS (lặp) có theo dõi cha

**Giải thích.** Xây danh sách kề; DFS từ nút `0` với `(nút, cha)`: gặp nút đã thăm khác cha ⇒ chu trình ⇒ `False`. Cuối cùng yêu cầu **thăm đủ `n` nút** (liên thông).
#### Code demo

In [10]:
def valid_tree_dfs(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    visited = [False] * n
    visited[0] = True
    count = 1
    stack = [(0, -1)]
    while stack:
        v, parent = stack.pop()
        for u in adj[v]:
            if u == parent:
                continue
            if visited[u]:
                return False                           # chu trình
            visited[u] = True
            count += 1
            stack.append((u, v))
    return count == n                                  # phải thăm đủ n nút (liên thông)


for n, edges, expected in examples:
    print(n, edges, "->", valid_tree_dfs(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [0, 2], [0, 3], [1, 4]] -> True (kỳ vọng True )
5 [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(V + E)`**, **Space `O(V + E)`** (danh sách kề, stack, `visited`). Bản đệ quy cũng được nhưng **với chuỗi dài `n = 2000` có thể vượt giới hạn đệ quy mặc định `1000`** của Python — bản lặp an toàn.

### Cách 4 — BFS kèm kiểm tra số cạnh

**Giải thích.** Nếu `len(edges) != n - 1` trả `False` ngay (rẻ nhất). Ngược lại chỉ cần kiểm tra **liên thông** bằng BFS từ `0`: với đúng `n - 1` cạnh, liên thông ⇒ cây.
#### Code demo

In [11]:
def valid_tree_bfs(n, edges):
    if len(edges) != n - 1:
        return False
    adj = defaultdict(list)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    seen = {0}
    queue = deque([0])
    while queue:
        v = queue.popleft()
        for u in adj[v]:
            if u not in seen:
                seen.add(u)
                queue.append(u)
    return len(seen) == n


for n, edges, expected in examples:
    print(n, edges, "->", valid_tree_bfs(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [0, 2], [0, 3], [1, 4]] -> True (kỳ vọng True )
5 [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> False (kỳ vọng False )


**Độ phức tạp Cách 4:** **Time `O(V + E)`** (và `O(1)` nếu số cạnh sai), **Space `O(V + E)`**.

### Cách 5 — Bóc lá (leaf peeling, kiểu Kahn)

**Giải thích.** Cây luôn có ít nhất một **lá** (nút bậc `<= 1`); bóc lá liên tục thì cây "tan" hết. Nếu có `n - 1` cạnh và quá trình **bóc được đủ `n` nút** thì đồ thị **không chu trình** (chu trình không bao giờ có nút bậc `1` nên không bóc được) ⇒ cây. Cài đặt: tính bậc, hàng đợi các nút bậc `<= 1`, mỗi lần bóc giảm bậc hàng xóm và đưa vào hàng đợi khi bậc về `1`.
#### Code demo

In [12]:
def valid_tree_peeling(n, edges):
    if len(edges) != n - 1:
        return False
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    degree = [len(adj[v]) for v in range(n)]
    queue = deque(v for v in range(n) if degree[v] <= 1)
    removed = 0
    while queue:
        v = queue.popleft()
        removed += 1
        for u in adj[v]:
            degree[u] -= 1
            if degree[u] == 1:
                queue.append(u)
    return removed == n


for n, edges, expected in examples:
    print(n, edges, "->", valid_tree_peeling(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [0, 2], [0, 3], [1, 4]] -> True (kỳ vọng True )
5 [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]] -> False (kỳ vọng False )


**Độ phức tạp Cách 5:** **Time `O(V + E)`**, **Space `O(V + E)`**. Đây là cách "không cần `visited` và không cần chọn gốc", cùng ý với thuật toán tìm gốc của **cây có chiều cao nhỏ nhất** (LeetCode 310).

> Cách 2 là bản nộp tiêu chuẩn của bài. Cách 3/4 cho thấy DFS/BFS. Cách 5 là góc nhìn "bóc lá".

## 8. TRACE từng bước

Ta trace Union-Find trên cả hai ví dụ (in từng cạnh, hai gốc, hành động và số thành phần), rồi trace DFS với cha trên ví dụ 2.
#### Code demo

In [13]:
def trace_dsu(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    print(f"n={n}, edges={edges}")
    components = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            print(f"  cạnh ({a},{b}): find({a})={ra} == find({b})={rb} -> CHU TRÌNH, trả False")
            return False
        parent[ra] = rb
        components -= 1
        print(f"  cạnh ({a},{b}): find({a})={ra} != find({b})={rb} -> gộp; parent={parent}; số thành phần={components}")
    print(f"  hết cạnh: số thành phần = {components} -> {'True' if components == 1 else 'False'}")
    return components == 1


trace_dsu(5, [[0, 1], [0, 2], [0, 3], [1, 4]])
print()
trace_dsu(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]])

n=5, edges=[[0, 1], [0, 2], [0, 3], [1, 4]]
  cạnh (0,1): find(0)=0 != find(1)=1 -> gộp; parent=[1, 1, 2, 3, 4]; số thành phần=4
  cạnh (0,2): find(0)=1 != find(2)=2 -> gộp; parent=[1, 2, 2, 3, 4]; số thành phần=3
  cạnh (0,3): find(0)=2 != find(3)=3 -> gộp; parent=[2, 2, 3, 3, 4]; số thành phần=2
  cạnh (1,4): find(1)=3 != find(4)=4 -> gộp; parent=[2, 3, 3, 4, 4]; số thành phần=1
  hết cạnh: số thành phần = 1 -> True

n=5, edges=[[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]
  cạnh (0,1): find(0)=0 != find(1)=1 -> gộp; parent=[1, 1, 2, 3, 4]; số thành phần=4
  cạnh (1,2): find(1)=1 != find(2)=2 -> gộp; parent=[1, 2, 2, 3, 4]; số thành phần=3
  cạnh (2,3): find(2)=2 != find(3)=3 -> gộp; parent=[1, 2, 3, 3, 4]; số thành phần=2
  cạnh (1,3): find(1)=3 == find(3)=3 -> CHU TRÌNH, trả False


False

In [14]:
def trace_dfs(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    print("danh sách kề:", adj)
    visited = [False] * n
    visited[0] = True
    stack = [(0, -1)]
    while stack:
        v, parent = stack.pop()
        print(f"  thăm {v} (cha {parent}), kề {adj[v]}")
        for u in adj[v]:
            if u == parent:
                print(f"    {u}: là cha, bỏ qua")
                continue
            if visited[u]:
                print(f"    {u}: đã thăm và không phải cha -> CHU TRÌNH")
                return False
            visited[u] = True
            stack.append((u, v))
            print(f"    {u}: chưa thăm -> đẩy ({u}, {v})")
    print("thăm được", sum(visited), "nút / n =", n)
    return sum(visited) == n


trace_dfs(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]])

danh sách kề: [[1], [0, 2, 3, 4], [1, 3], [2, 1], [1]]
  thăm 0 (cha -1), kề [1]
    1: chưa thăm -> đẩy (1, 0)
  thăm 1 (cha 0), kề [0, 2, 3, 4]
    0: là cha, bỏ qua
    2: chưa thăm -> đẩy (2, 1)
    3: chưa thăm -> đẩy (3, 1)
    4: chưa thăm -> đẩy (4, 1)
  thăm 4 (cha 1), kề [1]
    1: là cha, bỏ qua
  thăm 3 (cha 1), kề [2, 1]
    2: đã thăm và không phải cha -> CHU TRÌNH


False

Bảng tóm tắt DSU (ví dụ 2):

| cạnh | `find(a)` | `find(b)` | hành động | số thành phần |
|---|---|---|---|---|
| `(0,1)` | 0 | 1 | gộp | 4 |
| `(1,2)` | 1 | 2 | gộp | 3 |
| `(2,3)` | 2 | 3 | gộp | 2 |
| `(1,3)` | cùng gốc | cùng gốc | **chu trình** ⇒ `False` | — |

## 9. Code nộp LeetCode

Bản nộp dùng Union-Find (Cách 2) với `find` **lặp** (tránh tràn đệ quy khi chuỗi dài). Đúng `class Solution`, đúng `validTree(self, n: int, edges: List[List[int]]) -> bool`.

In [15]:
class Solution:
    def validTree(self, n: int, edges: List[List[int]]) -> bool:
        def find(x: int) -> int:
            while p[x] != x:                # tìm gốc, nén đường đi (path halving)
                p[x] = p[p[x]]
                x = p[x]
            return x

        p = list(range(n))
        for a, b in edges:
            pa, pb = find(a), find(b)
            if pa == pb:                    # đã liên thông: thêm cạnh này tạo chu trình
                return False
            p[pa] = pb                      # gộp hai thành phần
            n -= 1                          # n lúc này đếm số thành phần
        return n == 1


sol = Solution()
for n, edges, expected in examples:
    assert sol.validTree(n, edges) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Định lý cạnh–nút:** đồ thị vô hướng `n` nút là cây ⟺ không chu trình và liên thông. Mỗi cạnh thêm vào có hai khả năng: **nối hai thành phần khác nhau** (số thành phần giảm đúng `1`) hoặc **nối hai nút cùng thành phần** (tạo chu trình, số thành phần không đổi).
- **DSU:** `find(a) == find(b)` ⟺ `a`, `b` đã nằm cùng thành phần liên thông ⟺ cạnh `(a, b)` tạo chu trình. Nếu không cạnh nào tạo chu trình thì mỗi cạnh giảm số thành phần đi `1`; sau `E` cạnh còn `n - E` thành phần. **Liên thông ⟺ `n - E == 1` ⟺ `E == n - 1`.** Vì vậy "không chu trình" và `components == 1` cùng nhau đúng khi và chỉ khi đồ thị là cây.
- **DFS/BFS:** với đúng `n - 1` cạnh, liên thông ⇒ không thể có chu trình (nếu có, bỏ một cạnh trên chu trình vẫn liên thông với `n - 2` cạnh, mâu thuẫn với "liên thông cần `≥ n - 1` cạnh"). DFS có cha: nút đã thăm khác cha chứng tỏ chu trình; `count == n` chứng tỏ liên thông.
- **Bóc lá:** đồ thị có chu trình thì mọi nút trên chu trình luôn có bậc `≥ 2` nên không bao giờ thành lá ⇒ không bóc hết; rừng (không chu trình) luôn bóc hết. Cộng `n - 1` cạnh ⇒ rừng với đúng một thành phần ⇒ cây.
- **Biên:** `n = 1` không cạnh: `0 == n - 1` và DSU không gặp chu trình, `components == 1` ⇒ `True`.

## 11. Phân tích Time / Space Complexity

Gọi `V = n` (≤ 2000), `E = len(edges)` (≤ 5000), `α` là hàm Ackermann nghịch đảo.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cạnh là cầu | `O(E·(V + E))` | `O(V + E)` |
| 2. Union-Find | `O(E·α(V)) ≈ O(E)` | `O(V)` |
| 3. DFS lặp có cha | `O(V + E)` | `O(V + E)` |
| 4. BFS + đếm cạnh | `O(V + E)` | `O(V + E)` |
| 5. Bóc lá | `O(V + E)` | `O(V + E)` |

Với `V = 2000`, `E = 5000`: Cách 2–5 cỡ `10^4` phép — tức thời; Cách 1 cỡ `3.5·10^7`. Nhờ kiểm tra `E != V - 1` ở đầu, nhiều đầu vào trả về ngay trong `O(1)`.

## 12. Bộ test

`oracle` là brute force **độc lập**: tính **bao đóng bắc cầu (Floyd–Warshall)** để biết liên thông, và định nghĩa cây là "liên thông và **bỏ bất kỳ cạnh nào cũng mất liên thông**" (mọi cạnh là cầu) — không dùng DSU/DFS/đếm cạnh. Dùng cho `n <= 8`. Với `n` lớn, đối chiếu các cách với nhau và với đồ thị sinh theo cấu trúc (cây, cây thêm cạnh, rừng) có đáp án đã biết.
#### Code demo

In [16]:
def closure_connected(n, edge_list):
    reach = [[i == j for j in range(n)] for i in range(n)]
    for a, b in edge_list:
        reach[a][b] = reach[b][a] = True
    for k in range(n):
        for i in range(n):
            if reach[i][k]:
                for j in range(n):
                    if reach[k][j]:
                        reach[i][j] = True
    return all(reach[0][j] for j in range(n))


def oracle(n, edges):
    if not closure_connected(n, edges):
        return False
    for idx in range(len(edges)):
        if closure_connected(n, edges[:idx] + edges[idx + 1:]):    # bỏ cạnh mà vẫn liên thông: cạnh nằm trên chu trình
            return False
    return True


impls = {
    "bruteforce": valid_tree_bruteforce,
    "dsu": valid_tree_dsu,
    "dfs": valid_tree_dfs,
    "bfs": valid_tree_bfs,
    "peeling": valid_tree_peeling,
    "submit": lambda n, edges: sol.validTree(n, [list(e) for e in edges]),
}

for n, edges, expected in examples:
    assert oracle(n, edges) == expected
    for key, fn in impls.items():
        assert fn(n, [list(e) for e in edges]) == expected, (key, n, edges, fn(n, edges), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    (1, [], True, "một nút, không cạnh"),
    (2, [], False, "hai nút không cạnh (không liên thông)"),
    (2, [[0, 1]], True, "hai nút một cạnh"),
    (3, [[0, 1], [1, 2]], True, "đường đi"),
    (3, [[0, 1], [0, 2]], True, "sao"),
    (3, [[0, 1], [1, 2], [0, 2]], False, "tam giác: liên thông nhưng có chu trình"),
    (4, [[0, 1], [2, 3]], False, "hai đoạn rời: không chu trình nhưng không liên thông"),
    (4, [[0, 1], [1, 2], [2, 0]], False, "tam giác + nút cô lập (đủ n-1 = 3 cạnh nhưng không phải cây)"),
    (4, [[0, 1], [1, 2], [2, 3]], True, "đường dài"),
    (4, [[3, 2], [2, 1], [1, 0]], True, "cạnh ghi theo thứ tự/hướng đảo"),
    (5, [[0, 1], [0, 2], [0, 3], [1, 4]], True, "ví dụ 1"),
    (5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]], False, "ví dụ 2"),
    (6, [[0, 1], [1, 2], [2, 0], [3, 4], [4, 5]], False, "đủ n-1 = 5 cạnh nhưng gồm tam giác và đường rời"),
]
for n, edges, expected, label in edge_cases:
    assert oracle(n, edges) == expected, (label, oracle(n, edges), expected)
    for key, fn in impls.items():
        assert fn(n, [list(e) for e in edges]) == expected, (key, label, fn(n, edges), expected)
    print("PASS", label)

PASS một nút, không cạnh
PASS hai nút không cạnh (không liên thông)
PASS hai nút một cạnh
PASS đường đi
PASS sao
PASS tam giác: liên thông nhưng có chu trình
PASS hai đoạn rời: không chu trình nhưng không liên thông
PASS tam giác + nút cô lập (đủ n-1 = 3 cạnh nhưng không phải cây)
PASS đường dài
PASS cạnh ghi theo thứ tự/hướng đảo
PASS ví dụ 1
PASS ví dụ 2
PASS đủ n-1 = 5 cạnh nhưng gồm tam giác và đường rời


### 12.2. Tính chất và phản ví dụ

- Cây `n` nút luôn có đúng **`n - 1` cạnh**; thêm bất kỳ cạnh nào (không lặp) vào cây tạo **chu trình**; bỏ bất kỳ cạnh nào làm **mất liên thông**.
- **Bất biến** khi đổi nhãn nút, đổi thứ tự cạnh, đảo chiều từng cạnh `[a, b] ↔ [b, a]`.
- **Phản ví dụ** cho "chỉ kiểm tra chu trình" và "chỉ kiểm tra liên thông" (xem phần lỗi): `[[0,1],[2,3]]` (n=4) và `[[0,1],[1,2],[0,2]]` (n=3).
#### Code demo

In [18]:
def random_tree_edges(n, rng):
    labels = list(range(n))
    rng.shuffle(labels)
    edges = []
    for i in range(1, n):
        parent = labels[rng.randrange(i)]
        a, b = labels[i], parent
        edges.append([a, b] if rng.random() < 0.5 else [b, a])
    rng.shuffle(edges)
    return edges


rng = random.Random(261)
for _ in range(400):
    n = rng.randint(1, 30)
    edges = random_tree_edges(n, rng)
    assert len(edges) == n - 1 and sol.validTree(n, [e[:] for e in edges])
    # đảo chiều + xáo thứ tự + đổi nhãn vẫn là cây
    relabel = list(range(n))
    rng.shuffle(relabel)
    moved = [[relabel[a], relabel[b]] for a, b in edges]
    rng.shuffle(moved)
    assert sol.validTree(n, moved)
    if n >= 3:
        all_pairs = {tuple(sorted(e)) for e in edges}
        candidates = [(a, b) for a, b in combinations(range(n), 2) if (a, b) not in all_pairs]
        if candidates:
            extra = list(rng.choice(candidates))
            assert sol.validTree(n, [e[:] for e in edges] + [extra]) is False         # thêm cạnh: chu trình
        removed = edges[:]
        removed.pop(rng.randrange(len(removed)))
        assert sol.validTree(n, removed) is False                                    # bỏ cạnh: mất liên thông
print("PASS tính chất trên 400 cây ngẫu nhiên (đổi nhãn/thứ tự/hướng, thêm cạnh, bỏ cạnh)")


def sai_chi_kiem_tra_chu_trinh(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        parent[ra] = rb
    return True                                         # SAI: quên kiểm tra liên thông


def sai_chi_kiem_tra_lien_thong(n, edges):
    return connected(n, edges)                          # SAI: quên kiểm tra chu trình


print("n=4 [[0,1],[2,3]]: đúng", sol.validTree(4, [[0, 1], [2, 3]]), "| chỉ xét chu trình:", sai_chi_kiem_tra_chu_trinh(4, [[0, 1], [2, 3]]))
print("n=3 tam giác      : đúng", sol.validTree(3, [[0, 1], [1, 2], [0, 2]]), "| chỉ xét liên thông:", sai_chi_kiem_tra_lien_thong(3, [[0, 1], [1, 2], [0, 2]]))
assert sai_chi_kiem_tra_chu_trinh(4, [[0, 1], [2, 3]]) is True
assert sai_chi_kiem_tra_lien_thong(3, [[0, 1], [1, 2], [0, 2]]) is True

PASS tính chất trên 400 cây ngẫu nhiên (đổi nhãn/thứ tự/hướng, thêm cạnh, bỏ cạnh)
n=4 [[0,1],[2,3]]: đúng False | chỉ xét chu trình: True
n=3 tam giác      : đúng False | chỉ xét liên thông: True


### 12.3. Đối chiếu ngẫu nhiên (có seed) và đồ thị lớn
#### Code demo

In [19]:
rng = random.Random(4261)
trees_seen = 0
for trial in range(3000):
    n = rng.randint(1, 8)
    all_pairs = list(combinations(range(n), 2))
    max_edges = len(all_pairs)
    if rng.random() < 0.6:
        m = min(max(n - 1 + rng.randint(-1, 1), 0), max_edges)           # tập trung quanh n - 1
    else:
        m = rng.randint(0, max_edges)
    chosen = rng.sample(all_pairs, m)
    edges = [list(e) if rng.random() < 0.5 else [e[1], e[0]] for e in chosen]
    truth = oracle(n, edges)
    trees_seen += truth
    for key, fn in impls.items():
        assert fn(n, [list(e) for e in edges]) == truth, (key, n, edges, fn(n, edges), truth)
print(f"PASS 3000 đồ thị ngẫu nhiên n <= 8 (trong đó {trees_seen} cây): mọi cách khớp oracle")

# n lớn: cây (đường đi, sao, ngẫu nhiên) và đồ thị 'gần là cây'
big_cases = []
chain = [[i, i + 1] for i in range(1999)]
big_cases.append(("đường đi 2000 nút", 2000, chain, True))
big_cases.append(("đường đi, cạnh đảo thứ tự", 2000, chain[::-1], True))
big_cases.append(("sao 2000 nút", 2000, [[0, i] for i in range(1, 2000)], True))
tree = random_tree_edges(2000, rng)
big_cases.append(("cây ngẫu nhiên 2000 nút", 2000, tree, True))
existing = {tuple(sorted(e)) for e in tree}
extra_edge = next([a, b] for a, b in combinations(range(2000), 2) if (a, b) not in existing)
big_cases.append(("cây thêm một cạnh (chu trình)", 2000, tree + [extra_edge], False))
big_cases.append(("cây bỏ một cạnh", 2000, tree[:-1], False))
dense = set()
while len(dense) < 5000:
    a, b = rng.sample(range(2000), 2)
    dense.add((min(a, b), max(a, b)))
big_cases.append(("5000 cạnh ngẫu nhiên", 2000, [list(e) for e in dense], False))
for label, n, edges, expected in big_cases:
    row = []
    for key in ("dsu", "dfs", "bfs", "peeling", "submit"):
        t0 = time.perf_counter()
        value = impls[key](n, [list(e) for e in edges])
        dt = (time.perf_counter() - t0) * 1000
        assert value == expected, (label, key, value, expected)
        row.append(f"{key}={dt:5.1f}ms")
    print(f"{label:<32} -> {expected!s:<5} | " + " ".join(row))
print("PASS đồ thị lớn")

PASS 3000 đồ thị ngẫu nhiên n <= 8 (trong đó 934 cây): mọi cách khớp oracle


đường đi 2000 nút                -> True  | dsu=  0.5ms dfs=  1.9ms bfs=  0.7ms peeling=  0.9ms submit=  0.5ms
đường đi, cạnh đảo thứ tự        -> True  | dsu=  0.5ms dfs=  0.5ms bfs=  0.6ms peeling=  0.6ms submit=  0.7ms


sao 2000 nút                     -> True  | dsu=  0.5ms dfs=  0.6ms bfs=  0.6ms peeling=  1.1ms submit=  0.7ms
cây ngẫu nhiên 2000 nút          -> True  | dsu=  0.7ms dfs=  1.6ms bfs=  0.9ms peeling=  0.9ms submit=  0.8ms


cây thêm một cạnh (chu trình)    -> False | dsu=  1.0ms dfs=  0.7ms bfs=  0.2ms peeling=  0.2ms submit=  0.8ms
cây bỏ một cạnh                  -> False | dsu=  0.7ms dfs=  0.7ms bfs=  0.1ms peeling=  0.1ms submit=  0.8ms


5000 cạnh ngẫu nhiên             -> False | dsu=  0.7ms dfs=  1.0ms bfs=  0.4ms peeling=  0.4ms submit=  1.3ms
PASS đồ thị lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra chu trình (quên liên thông)
Xem `sai_chi_kiem_tra_chu_trinh` ở phần test: `n = 4`, `[[0,1],[2,3]]` không có chu trình nên bản sai trả `True`, nhưng đồ thị gồm **hai đoạn rời**. **Code đúng:** kiểm tra thêm liên thông (`components == 1` hoặc `visited == n` hoặc `len(edges) == n - 1`).

### Lỗi 2: Chỉ kiểm tra liên thông (quên chu trình)
Xem `sai_chi_kiem_tra_lien_thong`: tam giác `[[0,1],[1,2],[0,2]]` liên thông nên bản sai trả `True`. **Code đúng:** thêm kiểm tra chu trình hoặc `len(edges) == n - 1`.

### Lỗi 3: DFS vô hướng không loại cạnh về cha
#### Code sai

In [20]:
def sai_khong_loai_cha(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    visited = [False] * n
    visited[0] = True
    stack = [0]
    count = 1
    while stack:
        v = stack.pop()
        for u in adj[v]:
            if visited[u]:
                return False                              # SAI: cả cha cũng "đã thăm" nên cây nào cũng bị báo có chu trình
            visited[u] = True
            count += 1
            stack.append(u)
    return count == n


print("n=2 [[0,1]]: đúng", sol.validTree(2, [[0, 1]]), "| không loại cha:", sai_khong_loai_cha(2, [[0, 1]]))
print("ví dụ 1    : đúng", sol.validTree(*examples[0][:2]), "| không loại cha:", sai_khong_loai_cha(*examples[0][:2]))

n=2 [[0,1]]: đúng True | không loại cha: False
ví dụ 1    : đúng True | không loại cha: False


**Vì sao sai:** trong đồ thị vô hướng, từ `u` ta luôn thấy lại `v` (cha) trong `adj[u]` — đó là chính cạnh vừa đi, **không phải chu trình**. Phải lưu `parent` và bỏ qua `u == parent`.

### Lỗi 4: Chỉ thêm cạnh **một chiều** vào danh sách kề
#### Code sai

In [21]:
def sai_mot_chieu(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)                                  # SAI: thiếu adj[b].append(a)
    visited = [False] * n
    visited[0] = True
    stack, count = [0], 1
    while stack:
        v = stack.pop()
        for u in adj[v]:
            if not visited[u]:
                visited[u] = True
                count += 1
                stack.append(u)
    return count == n and len(edges) == n - 1


print("[[1,0],[1,2]] n=3 (cạnh ghi ngược): đúng", sol.validTree(3, [[1, 0], [1, 2]]), "| một chiều:", sai_mot_chieu(3, [[1, 0], [1, 2]]))

[[1,0],[1,2]] n=3 (cạnh ghi ngược): đúng True | một chiều: False


**Vì sao sai:** đồ thị **vô hướng**: cạnh `[1, 0]` nối cả `0 → 1` lẫn `1 → 0`. Chỉ thêm `adj[1].append(0)` thì từ `0` không đi được tới `1`.

### Lỗi 5: Quên kiểm tra `visited == n` (chỉ DFS từ nút `0`)
#### Code sai

In [22]:
def sai_quen_dem_nut(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    visited = [False] * n
    visited[0] = True
    stack = [(0, -1)]
    while stack:
        v, parent = stack.pop()
        for u in adj[v]:
            if u == parent:
                continue
            if visited[u]:
                return False
            visited[u] = True
            stack.append((u, v))
    return True                                           # SAI: không kiểm tra đã thăm đủ n nút


print("n=4 [[0,1]] (nút 2, 3 cô lập): đúng", sol.validTree(4, [[0, 1]]), "| quên đếm:", sai_quen_dem_nut(4, [[0, 1]]))

n=4 [[0,1]] (nút 2, 3 cô lập): đúng False | quên đếm: True


**Vì sao sai:** DFS từ `0` chỉ thăm thành phần chứa `0`; các nút cô lập không được thăm nhưng bản sai không phát hiện. **Code đúng:** `return count == n`.

### Lỗi 6: `find` đệ quy bị tràn khi chuỗi dài
#### Code sai

In [23]:
def valid_tree_dsu_recursive(n, edges):
    p = list(range(n))

    def find(x):
        if p[x] != x:
            p[x] = find(p[x])                              # đệ quy: chuỗi dài làm vượt giới hạn đệ quy
        return p[x]

    for a, b in edges:
        pa, pb = find(a), find(b)
        if pa == pb:
            return False
        p[pa] = pb
        n -= 1
    return n == 1


deep_tree = [[i, i + 1] for i in range(1998)] + [[1999, 0]]     # đường đi 0-1-...-1998 rồi gắn nút 1999 vào nút 0: vẫn là cây
print("số cạnh:", len(deep_tree), "= n - 1 =", 2000 - 1)
try:
    print("find đệ quy:", valid_tree_dsu_recursive(2000, deep_tree))
except RecursionError as e:
    print("RecursionError:", str(e)[:60], "-> cây DSU sâu ~2000 tầng vượt giới hạn 1000")
print("find lặp   :", sol.validTree(2000, deep_tree))

số cạnh: 1999 = n - 1 = 1999
RecursionError: maximum recursion depth exceeded -> cây DSU sâu ~2000 tầng vượt giới hạn 1000
find lặp   : True


**Vì sao sai:** thứ tự nối cạnh có thể tạo một "cây DSU" rất sâu: nối `(0,1)`, `(1,2)`, ..., mỗi lần gốc cũ bị treo dưới nút mới nên đường từ `0` lên gốc dài dần tới `~n = 2000`; khi cạnh cuối `[1999, 0]` gọi `find(0)`, `find` đệ quy vượt giới hạn `1000` và ném `RecursionError`. **Cách sửa:** `find` **lặp** với nén đường đi (như bản nộp), hoặc **hợp nhất theo kích thước/hạng** (union by size/rank), hoặc nâng `sys.setrecursionlimit`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**có phải là cây không**" (đồ thị vô hướng) → **`len(edges) == n - 1` + liên thông** (hoặc không chu trình). Kiểm tra số cạnh **trước** để loại nhanh.
- Thấy "**phát hiện chu trình khi thêm cạnh / đồ thị tăng dần**" → **Union-Find**: `find(a) == find(b)` ⇒ chu trình.
- DFS vô hướng: nhớ **cha** để không coi cạnh vừa đi là chu trình; cuối nhớ kiểm tra **đã thăm đủ `n` nút**.
- Danh sách kề vô hướng: **hai chiều** (`adj[a].append(b); adj[b].append(a)`).
- `n` lớn chuỗi dài: cẩn thận **giới hạn đệ quy**; dùng bản **lặp**.
- Họ hàng: **Number of Connected Components** (323), **Redundant Connection** (684), **Minimum Height Trees** (310), **Course Schedule** (207, đồ thị có hướng).
- Luôn thử `n = 1` không cạnh, đồ thị rời, tam giác, đường dài, và đồ thị đủ `n - 1` cạnh nhưng **không** phải cây (tam giác + đoạn rời).

## 15. Biến thể của bài

1. **Number of Connected Components in an Undirected Graph** (LeetCode 323): đếm số thành phần liên thông (DSU/DFS).
2. **Redundant Connection** (684): tìm cạnh làm đồ thị thành có chu trình (cạnh gây `find(a) == find(b)` đầu tiên).
3. **Minimum Height Trees** (310): bóc lá để tìm gốc có chiều cao nhỏ nhất.
4. **Course Schedule** (207): phát hiện chu trình trong đồ thị **có hướng** (Kahn / DFS ba màu).
5. **Trả về cạnh dư** hoặc **số cạnh cần thêm/bớt** để thành cây: `components - 1` cạnh thêm và `E - (V - components)` cạnh thừa.
#### Code demo

In [24]:
def count_components(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    components = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            components -= 1
    return components


def redundant_edge(n, edges):
    """Cạnh đầu tiên (theo thứ tự) nối hai nút đã liên thông."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]
        parent[ra] = rb
    return None


def edges_to_make_tree(n, edges):
    """(số cạnh cần thêm, số cạnh cần bớt) để thành cây: thêm components - 1, bớt E - (n - components)."""
    comps = count_components(n, edges)
    return comps - 1, len(edges) - (n - comps)


print(count_components(4, [[0, 1], [2, 3]]), redundant_edge(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]), edges_to_make_tree(6, [[0, 1], [1, 2], [2, 0], [3, 4], [4, 5]]))
# 2 [1, 3] (1, 1): ví dụ cuối cần bớt 1 cạnh trong tam giác và thêm 1 cạnh nối hai thành phần

2 [1, 3] (1, 1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cạnh là cầu | liên thông và bỏ cạnh nào cũng mất liên thông | `O(E·(V+E))` | `O(V+E)` | oracle / hiểu định nghĩa |
| 2. Union-Find | `find(a) == find(b)` ⇒ chu trình; đếm thành phần | `O(E·α(V))` | `O(V)` | **cách nộp chuẩn**, đồ thị tăng dần |
| 3. DFS có cha | phát hiện chu trình + đếm nút thăm | `O(V+E)` | `O(V+E)` | khi cần duyệt cây thật sự |
| 4. BFS + đếm cạnh | `E == V-1` rồi kiểm tra liên thông | `O(V+E)` | `O(V+E)` | loại nhanh khi số cạnh sai |
| 5. Bóc lá | bóc nút bậc ≤ 1 cho tới khi hết | `O(V+E)` | `O(V+E)` | không cần gốc/`visited`; liên hệ MHT |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().validTree(1, [])`; (b) `Solution().validTree(3, [[0, 1]])`; (c) `Solution().validTree(4, [[0, 1], [1, 2], [2, 3]])`; (d) `Solution().validTree(4, [[0, 1], [1, 2], [0, 2]])`.

**Bài 2 (tính bằng tay).** Với `n = 6`, `edges = [[0,1],[0,2],[2,3],[2,4],[4,5]]`: vẽ đồ thị, đếm cạnh so với `n - 1`, và chạy tay Union-Find (ghi gốc và số thành phần sau mỗi cạnh).

**Bài 3 (điền TODO).** Hoàn thiện `valid_tree_todo(n, edges)` bằng Union-Find.
#### Code demo

In [25]:
def valid_tree_todo(n, edges):
    parent = list(range(n))

    def find(x):
        # TODO 1: while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]; rồi trả x
        pass

    components = n
    # TODO 2: duyệt từng cạnh (a, b); ra = find(a), rb = find(b)
    # TODO 3: nếu ra == rb -> trả False (chu trình)
    # TODO 4: ngược lại parent[ra] = rb; components -= 1
    # TODO 5: trả components == 1
    pass


# Tự làm xong rồi mới mở test:
# for n, edges, expected in examples:
#     assert valid_tree_todo(n, edges) == expected

**Bài 4 (tự viết lại).** Viết `valid_tree_dfs_recursive(n, edges)` bằng **DFS đệ quy** có tham số `parent` (và `sys.setrecursionlimit` đủ lớn), rồi so sánh với `oracle` cho `n <= 8`.

**Bài 5 (biến thể / mini challenge).** Viết `min_changes_to_tree(n, edges)` trả về **tổng số cạnh phải thêm và bớt tối thiểu** để đồ thị trở thành cây (dùng `edges_to_make_tree`), và `find_cycle_edge_order(n, edges)` trả về **chỉ số** của cạnh đầu tiên tạo chu trình (hoặc `-1`). Thử `n = 5`, `edges = [[0,1],[1,2],[2,3],[1,3],[1,4]]` → chỉ số `3`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: nhớ điều kiện `n - 1` cạnh và liên thông. Bài 2: `6` nút cần `5` cạnh. Bài 5: dùng `find(a) == find(b)` để phát hiện cạnh tạo chu trình.
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `True`, `False` (tam giác + nút `3` cô lập). Bài 2: có `5 = n - 1` cạnh; DSU: `(0,1)` gộp → 5 thành phần; `(0,2)` → 4; `(2,3)` → 3; `(2,4)` → 2; `(4,5)` → 1 ⇒ `True`. Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: `dfs(v, parent)`: với `u` trong `adj[v]`, `u == parent` thì bỏ qua; `visited[u]` thì chu trình; ngược lại đệ quy. Bài 5: thêm `components - 1` và bớt `E - (n - components)`; cạnh đầu tiên tạo chu trình: duyệt cạnh theo thứ tự với DSU, trả `index` ngay khi `find(a) == find(b)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
import sys

# Bài 1
assert Solution().validTree(1, []) is True
assert Solution().validTree(3, [[0, 1]]) is False
assert Solution().validTree(4, [[0, 1], [1, 2], [2, 3]]) is True
assert Solution().validTree(4, [[0, 1], [1, 2], [0, 2]]) is False

# Bài 2
b2 = [[0, 1], [0, 2], [2, 3], [2, 4], [4, 5]]
assert len(b2) == 6 - 1 and Solution().validTree(6, b2) is True and oracle(6, b2) is True


# Bài 3
def valid_tree_todo_ans(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    components = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        parent[ra] = rb
        components -= 1
    return components == 1


for n, edges, expected in examples:
    assert valid_tree_todo_ans(n, edges) == expected


# Bài 4
def valid_tree_dfs_recursive(n, edges):
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 10_000))
    try:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        visited = [False] * n
        has_cycle = False

        def dfs(v, parent):
            nonlocal has_cycle
            visited[v] = True
            for u in adj[v]:
                if u == parent:
                    continue
                if visited[u]:
                    has_cycle = True
                else:
                    dfs(u, v)

        dfs(0, -1)
        return not has_cycle and all(visited)
    finally:
        sys.setrecursionlimit(old)


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 8)
    pairs = list(combinations(range(n), 2))
    m = rng.randint(0, len(pairs))
    edges = [list(e) for e in rng.sample(pairs, m)]
    assert valid_tree_dfs_recursive(n, edges) == oracle(n, edges)


# Bài 5
def min_changes_to_tree(n, edges):
    add, remove = edges_to_make_tree(n, edges)
    return add + remove


def find_cycle_edge_order(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for idx, (a, b) in enumerate(edges):
        ra, rb = find(a), find(b)
        if ra == rb:
            return idx
        parent[ra] = rb
    return -1


assert find_cycle_edge_order(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]) == 3
assert find_cycle_edge_order(5, [[0, 1], [0, 2], [0, 3], [1, 4]]) == -1
assert min_changes_to_tree(6, [[0, 1], [1, 2], [2, 0], [3, 4], [4, 5]]) == 2
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Union-Find, `find` lặp)

```python
class Solution:
    def validTree(self, n: int, edges: List[List[int]]) -> bool:
        def find(x):
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        p = list(range(n))
        for a, b in edges:
            pa, pb = find(a), find(b)
            if pa == pb:
                return False
            p[pa] = pb
            n -= 1
        return n == 1
```

**Checklist**

- [ ] Cây = **liên thông** **và** **không chu trình** (⟺ liên thông và `n - 1` cạnh)? Đã kiểm tra cả hai (hoặc `len(edges) == n - 1` + một trong hai)?
- [ ] Danh sách kề **hai chiều**; DFS vô hướng **loại cạnh về cha**?
- [ ] Sau DFS/BFS: **thăm đủ `n` nút**?
- [ ] DSU: `find(a) == find(b)` ⇒ `False`; `union` giảm số thành phần; cuối `components == 1`?
- [ ] `find` **lặp** (hoặc nâng giới hạn đệ quy) để chuỗi dài `n = 2000` không tràn?
- [ ] Đã thử `n = 1` không cạnh, hai đoạn rời, tam giác, tam giác + nút cô lập (đủ `n - 1` cạnh)?
- [ ] Đã đối chiếu với brute force "liên thông và mọi cạnh là cầu" trên đồ thị ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def validTree(self, n: int, edges: List[List[int]]) -> bool`?